# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/github/AmanDbz1101/FlyRank-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)]

**Lane: Refresh / Content Opportunity Scoring (Lane 2).** This notebook writes the data contract for one month's slice of the search river — the daily facts that flow from Search Console / Analytics into the warehouse — proves three facts about it with real queries, builds five features, and runs the leakage trap from notebook 02 on real warehouse rows. Then it keeps the honest number.

> Worked from `skills/writing-data-contracts/SKILL.md` + `skills/flyrank/flyrank-data/SKILL.md`. One task: write + verify the contract on a mid-panel month, `month=2026-03`.

## 1. Unit of analysis + time window

**The contract, in plain words (5 answers):**

1. **What one row means.** One row = **one content page** (`content_hash_id`) as of a single decision moment. The daily fact rows for that page are aggregated over the decision window, so one row = one page, not one page-day.
2. **Which tables.** `fact_content_daily_performance` (months `2026-02`, `2026-03`, `2026-04`) for the signals and the outcome; `dim_content` for page context (type, keyword, age); `dim_clients` for history-coverage context. That is my lane's slice of the river.
3. **Which time window.** Feature window = **March 2026** (`2026-03-01` → `2026-03-31`). Decision moment = **end of March, `2026-03-31`**. Label window = **April 2026** (the next 30 days). February is read only to build one momentum feature that is fully knowable at the decision moment.
4. **What we predict / rank.** Whether a *visible* page's search impressions fall in the next 30 days. Proxy label: `is_declining = 1` when April impressions < 80% of March impressions, with a minimum-volume floor (March impressions ≥ 30) so a low-volume wiggle is not counted as decline. Pages are **ranked by that risk** so an editor reviews the most at-risk pages first.
5. **One thing I deliberately exclude.** `fact_content_query_90d` — its single fixed 90-day window overlaps the March/April label months, so its columns would leak the answer; its per-content context also repeats on every row (a `SUM()` trap). It lives in section 2's excluded bucket.

In [1]:
# SETUP — the token stays in the runtime, never in a cell (this repo is public).
import os, sys, subprocess, importlib.util, warnings
warnings.filterwarnings("ignore", message="IProgress not found")

IN_COLAB = "google.colab" in sys.modules
if any(importlib.util.find_spec(m) is None for m in ("duckdb", "sklearn", "huggingface_hub")):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "duckdb", "scikit-learn", "huggingface_hub", "pandas", "numpy"], check=True)

import duckdb, numpy as np, pandas as pd
from huggingface_hub import hf_hub_download, get_token

assert get_token(), ("No read token found. In Colab add a read-only HF_TOKEN as a Secret "
                     "(key panel) and rerun; the token is read from the runtime, not typed here.")

DS = "FlyRank/internship-warehouse"
P = {m: hf_hub_download(DS, f"fact_content_daily_performance/month={m}/data_0.parquet", repo_type="dataset")
     for m in ["2026-02", "2026-03", "2026-04"]}
DIM_CONTENT = hf_hub_download(DS, "dim_content.parquet", repo_type="dataset")
DIM_CLIENTS = hf_hub_download(DS, "dim_clients.parquet", repo_type="dataset")
con = duckdb.connect()

# ---- Backs section 1: the raw grain of the fact table and the slice window.
MAR = f"read_parquet('{P['2026-03']}')"
grain = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {MAR}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
""").df()
print("Raw-table grain probe (report_date x client x content): duplicates =", len(grain))

w = con.execute(f"SELECT MIN(report_date) AS lo, MAX(report_date) AS hi FROM {MAR}").df()
print(f"Slice window: {w['lo'][0]}  ->  {w['hi'][0]}")

print("\nMy unit of analysis collapses those daily rows into ONE row per content page;")
print("the aggregation grain is verified in section 3, query 1.")

Raw-table grain probe (report_date x client x content): duplicates = 0
Slice window: 2026-03-01 00:00:00  ->  2026-03-31 00:00:00

My unit of analysis collapses those daily rows into ONE row per content page;
the aggregation grain is verified in section 3, query 1.


## 2. Fields — feature / label / context / excluded

Every field this contract touches goes into exactly one bucket. The code cell prints the concrete column list.

| Bucket | Fields | One-line why |
|---|---|---|
| **Feature** (knowable at `2026-03-31`) | `momentum_feb_to_mar_pct`, `log_impressions_mar`, `ctr_mar`, `avg_position_mar`, `engagement_rate_mar` | Observed over windows that end 31 Mar — already true before anyone predicts April. |
| **Label / proxy** | `is_declining` = April < 80% of March | The future outcome we rank; **never** a feature. |
| **Context** | `content_hash_id`, `client_hash_id`, `content_type`, `search_volume`, `content_created_date` | Join / group / explain / split only — IDs and metadata are not learned-from features. |
| **Excluded** | `fact_content_query_90d`; April & later columns; `trend_*` buckets; raw query/URL/title | Query table's 90-day window overlaps the label months (leak); future columns contain the answer; `trend_*` is the starter label source; raw text is private and not shipped. |

In [2]:
# Backs section 2: the exact fields this notebook builds, classified.
FEATURES = {
    "momentum_feb_to_mar_pct": "feature (Feb->Mar, closed months)",
    "log_impressions_mar":     "feature (March GSC impressions, log1p)",
    "ctr_mar":                 "feature (March clicks / impressions x100)",
    "avg_position_mar":        "feature (March mean gsc_avg_position)",
    "engagement_rate_mar":     "feature (March GA4 engaged / sessions, IS TRUE rows)",
}
LABEL = {"is_declining": "label / proxy (April < 80% of March, floor >= 30)"}
CONTEXT = {"content_hash_id": "context (group/join/split only)",
           "client_hash_id":  "context (grouped splits only)",
           "content_type":    "context (read, not learned-from)",
           "search_volume":   "context (read, not learned-from)"}

print("Feature bucket:")
for k, v in FEATURES.items():
    print(f"  {k:26s} {v}")
print("Label bucket:")
for k, v in LABEL.items():
    print(f"  {k:26s} {v}")
print("Context bucket:")
for k, v in CONTEXT.items():
    print(f"  {k:26s} {v}")
print("Excluded: fact_content_query_90d, April/future columns, trend buckets, raw text")

Feature bucket:
  momentum_feb_to_mar_pct    feature (Feb->Mar, closed months)
  log_impressions_mar        feature (March GSC impressions, log1p)
  ctr_mar                    feature (March clicks / impressions x100)
  avg_position_mar           feature (March mean gsc_avg_position)
  engagement_rate_mar        feature (March GA4 engaged / sessions, IS TRUE rows)
Label bucket:
  is_declining               label / proxy (April < 80% of March, floor >= 30)
Context bucket:
  content_hash_id            context (group/join/split only)
  client_hash_id             context (grouped splits only)
  content_type               context (read, not learned-from)
  search_volume              context (read, not learned-from)
Excluded: fact_content_query_90d, April/future columns, trend buckets, raw text


## 3. Verify it with queries (grain, counts, availability) + the five features + the trap

Three verification queries on the mid-panel month `month=2026-03`, then the feature frame, then the deliberate-leak experiment.

In [3]:
# Q1. GRAIN — one aggregated row really is one content page.
agg = con.execute(f"""
    SELECT content_hash_id
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()
print("Aggregated frame rows:", len(agg), "| distinct content page ids:", agg['content_hash_id'].nunique())
print("Frame grain holds (1 page = 1 row):", len(agg) == agg['content_hash_id'].nunique())

Aggregated frame rows: 176738 | distinct content page ids: 176738
Frame grain holds (1 page = 1 row): True


In [4]:
# Q2. ROW COUNT + DATE SPAN of the slice (month=2026-03).
s = con.execute(f"""
    SELECT COUNT(*)                        AS rows_in_slice,
           COUNT(DISTINCT content_hash_id) AS content_pages,
           COUNT(DISTINCT client_hash_id)  AS clients,
           MIN(report_date)                AS span_start,
           MAX(report_date)                AS span_end
    FROM {MAR}
""").df().iloc[0]
print(f"Slice rows : {s['rows_in_slice']:,}")
print(f"Pages      : {s['content_pages']:,}")
print(f"Clients    : {s['clients']}")
print(f"Date span  : {s['span_start']}  ->  {s['span_end']}  (one full month)")

Slice rows : 9,841,378
Pages      : 331,437
Clients    : 55
Date span  : 2026-03-01 00:00:00  ->  2026-03-31 00:00:00  (one full month)


In [5]:
# Q3. AVAILABILITY — filter with IS TRUE and show how many rows survive.
# The availability flag is THREE-valued (TRUE / FALSE / NULL), so `IS TRUE` is the honest filter.
av = con.execute(f"""
    SELECT COUNT(*)                                  AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS survive_ga4_true,
           COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_false,
           COUNT(*) FILTER (WHERE ga4_data_available IS NULL)  AS ga4_null,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS gsc_true
    FROM {MAR}
""").df().iloc[0]
total = int(av['total_rows']); keep = int(av['survive_ga4_true'])
print(f"March slice rows: {total:,}")
print(f"  ga4_data_available IS TRUE : {keep:,}  ({keep/total:.1%}  -> survives the filter)")
print(f"  ga4_data_available IS FALSE: {av['ga4_false']:,}")
print(f"  ga4_data_available IS NULL : {av['ga4_null']:,}  (neither TRUE nor FALSE — also filtered out)")
print(f"  gsc_data_available IS TRUE : {av['gsc_true']:,}")
print(f"\nRows that survive 'IS TRUE': {keep:,} of {total:,} ({keep/total:.1%}).")
print("The other ~96% are pages/clients that had no GA4 mark yet — NOT 'zero engagement'.")

March slice rows: 9,841,378
  ga4_data_available IS TRUE : 413,966  (4.2%  -> survives the filter)
  ga4_data_available IS FALSE: 6,408,671
  ga4_data_available IS NULL : 3,018,741  (neither TRUE nor FALSE — also filtered out)
  gsc_data_available IS TRUE : 3,611,061

Rows that survive 'IS TRUE': 413,966 of 9,841,378 (4.2%).
The other ~96% are pages/clients that had no GA4 mark yet — NOT 'zero engagement'.


**The five features, each with its "available when?" line (decision moment = `2026-03-31`):**

1. `momentum_feb_to_mar_pct` — knowable at the decision moment **because** February and March are both closed history by 31 Mar; it is the Feb→Mar impression change computed only from finished months.
2. `log_impressions_mar` — knowable at the decision moment **because** March's daily Search Console impressions are all written before April begins.
3. `ctr_mar` — knowable at the decision moment **because** March clicks ÷ March impressions are both final at month end.
4. `avg_position_mar` — knowable at the decision moment **because** it is the mean of March's daily `gsc_avg_position`, which stops accruing on 31 Mar.
5. `engagement_rate_mar` — knowable at the decision moment **because** it sums March GA4 engaged sessions ÷ sessions only on rows where `ga4_data_available IS TRUE`; pages without a GA4 mark get 0 (see availability: ~96% of slice rows).

In [6]:
# Build the feature frame: one row per page, features from Mar, label from Apr.
FACT3 = "[" + ",".join(f"'{P[m]}'" for m in ["2026-02", "2026-03", "2026-04"]) + "]"
frame = con.execute(f"""
    SELECT
        f.content_hash_id,
        MAX(f.client_hash_id) AS client_hash_id,
        SUM(f.gsc_impressions) FILTER (WHERE f.month='2026-02' AND f.gsc_data_available IS TRUE) AS feb_impressions,
        SUM(f.gsc_impressions) FILTER (WHERE f.month='2026-03' AND f.gsc_data_available IS TRUE) AS mar_impressions,
        SUM(f.gsc_clicks)     FILTER (WHERE f.month='2026-03' AND f.gsc_data_available IS TRUE) AS mar_clicks,
        AVG(f.gsc_avg_position) FILTER (WHERE f.month='2026-03' AND f.gsc_data_available IS TRUE) AS mar_avg_position,
        SUM(f.ga4_engaged_sessions) FILTER (WHERE f.month='2026-03' AND f.ga4_data_available IS TRUE) AS mar_engaged,
        SUM(f.ga4_sessions)         FILTER (WHERE f.month='2026-03' AND f.ga4_data_available IS TRUE) AS mar_sessions,
        SUM(f.gsc_impressions) FILTER (WHERE f.month='2026-04' AND f.gsc_data_available IS TRUE) AS apr_impressions
    FROM read_parquet({FACT3}) AS f
    GROUP BY f.content_hash_id
""").df()

fr = frame.copy()
fr["log_impressions_mar"]     = np.log1p(fr["mar_impressions"].fillna(0))
fr["ctr_mar"]                 = np.where(fr["mar_impressions"].fillna(0) > 0,
                                         fr["mar_clicks"].fillna(0) / fr["mar_impressions"].replace(0, np.nan) * 100, 0.0)
fr["avg_position_mar"]        = fr["mar_avg_position"].fillna(0)          # 0 = "no position data", never rank 0
fr["momentum_feb_to_mar_pct"] = np.where(fr["feb_impressions"].fillna(0) > 0,
                                         (fr["mar_impressions"] - fr["feb_impressions"]) / fr["feb_impressions"] * 100, 0.0)
fr["engagement_rate_mar"]     = np.where(fr["mar_sessions"].fillna(0) > 0,
                                         fr["mar_engaged"].fillna(0) / fr["mar_sessions"].replace(0, np.nan) * 100, 0.0)
fr["is_declining"]            = (fr["apr_impressions"].fillna(0) < 0.8 * fr["mar_impressions"].fillna(0)).astype(int)

FEATURES5 = ["momentum_feb_to_mar_pct", "log_impressions_mar", "ctr_mar",
             "avg_position_mar", "engagement_rate_mar"]

# Honest modelling frame: visible pages only (volume floor: March impressions >= 30).
frame_ok = fr[fr["mar_impressions"].fillna(0) >= 30].copy()
print("Feature-frame rows (pages):", len(frame))
print("After volume floor (Mar impressions >= 30):", len(frame_ok), "| decline rate:",
      round(frame_ok["is_declining"].mean(), 3))
print("\nFirst rows of the frame:")
print(frame_ok[["content_hash_id", "client_hash_id"] + FEATURES5 + ["is_declining"]]
      .head().to_string(index=False))

Feature-frame rows (pages): 380147
After volume floor (Mar impressions >= 30): 125645 | decline rate: 0.518

First rows of the frame:
         content_hash_id          client_hash_id  momentum_feb_to_mar_pct  log_impressions_mar  ctr_mar  avg_position_mar  engagement_rate_mar  is_declining
content_9c57aacf529b2838 client_62f4a7e64f5e0096               -23.648649             4.736198 0.000000         15.197937             0.000000             1
content_116e12ab9a6de297 client_62f4a7e64f5e0096                 3.144654             6.892642 0.000000          6.827565             0.000000             1
content_e1ad38ef954fbb13 client_62f4a7e64f5e0096               135.616438             5.153292 0.000000          6.341944             0.000000             1
content_810cf06597918291 client_9958f0a7ae1df715                 0.761421             5.986452 0.251889         10.570485            14.285714             1
content_1b9a4c1e81e20eca client_9958f0a7ae1df715               800.000000        

In [7]:
# THE TRAP — add ONE label-window column on purpose, watch the score jump, then remove it.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

# Deterministic row order so the split (and every number below) is reproducible run-to-run.
frame_ok = frame_ok.sort_values("content_hash_id").reset_index(drop=True)
X = frame_ok[FEATURES5].fillna(0).values
y = frame_ok["is_declining"].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
sc = StandardScaler().fit(Xtr)
lr = LogisticRegression(max_iter=1000).fit(sc.transform(Xtr), ytr)
honest_auc = roc_auc_score(yte, lr.predict_proba(sc.transform(Xte))[:, 1])

# Deliberate leak: a column derived from the LABEL window (April) — looks innocent in the table.
leak = frame_ok.copy()
leak["mar_to_apr_change_pct"] = np.where(leak["mar_impressions"] > 0,
                                         (leak["apr_impressions"] - leak["mar_impressions"]) / leak["mar_impressions"] * 100, 0.0)
Xl = leak[FEATURES5 + ["mar_to_apr_change_pct"]].fillna(0).values
Xltr, Xlte, yltr, ylte = train_test_split(Xl, y, test_size=0.3, random_state=0, stratify=y)
scl = StandardScaler().fit(Xltr)
lrl = LogisticRegression(max_iter=1000).fit(scl.transform(Xltr), yltr)
leak_auc = roc_auc_score(ylte, lrl.predict_proba(scl.transform(Xlte))[:, 1])

print(f"Honest 5-feature model       ROC AUC = {honest_auc:.3f}")
print(f"After adding ONE label-window column  ROC AUC = {leak_auc:.3f}   << the trap")
print(f"\nThat future-window column must be dropped. Honest number kept: ROC AUC = {honest_auc:.3f}")

Honest 5-feature model       ROC AUC = 0.583
After adding ONE label-window column  ROC AUC = 0.993   << the trap

That future-window column must be dropped. Honest number kept: ROC AUC = 0.583


## 4. Data limits

**Named limitation of this slice: the March window only sees pages that already had real search data in March — an unbalanced panel.** `dim_clients` records **104 clients**, but only 67 carry a GSC start date (`2025-01-27` … `2026-06-02`) and just **55 clients have any March 2026 rows**. Pages and clients that began tracking later are *absent* from the frame, not measured as zero. So this contract describes the visible, tracked population of March 2026, not the whole inventory.

It also cannot separate *why* a page declined: a March→April drop can be real decay, seasonality, or SERP/AI-share churn, and a single comparison window cannot tell those apart — telling them needs a persistence window, which later weeks add on purpose. The label is a decision-support proxy, not a causal claim.

In [8]:
# Backs section 4: the unbalanced panel behind the March slice.
lim = con.execute(f"""
    SELECT
        (SELECT COUNT(*) FROM read_parquet('{DIM_CLIENTS}'))                                  AS clients_in_dim,
        (SELECT COUNT(*) FROM read_parquet('{DIM_CLIENTS}') WHERE gsc_data_start IS NOT NULL) AS clients_with_gsc_start,
        (SELECT COUNT(DISTINCT client_hash_id) FROM {MAR})                                    AS clients_active_in_march
""").df()
print(lim.to_string(index=False))
g = con.execute(f"""
    SELECT COUNT(*) AS n, MIN(gsc_data_start) AS earliest, MAX(gsc_data_start) AS latest
    FROM read_parquet('{DIM_CLIENTS}') WHERE gsc_data_start IS NOT NULL
""").df()
print(f"GSC start date range across {g['n'][0]} tracked clients: {g['earliest'][0]} -> {g['latest'][0]}")
print("History depth differs wildly per client; the frame is only as deep as each client's tracking.")

 clients_in_dim  clients_with_gsc_start  clients_active_in_march
            104                      67                       55
GSC start date range across 67 tracked clients: 2025-01-27 00:00:00 -> 2026-06-02 00:00:00
History depth differs wildly per client; the frame is only as deep as each client's tracking.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Contract answers in 1)–2): one row meaning, tables, window, label, one exclusion — each backed by a query
- [x] Exactly three verification queries in 3) with outputs visible; availability filtered with `IS TRUE`
- [x] Five-feature frame with an "available when?" line per feature; the deliberate-leak experiment shown and removed; honest number kept
- [x] One named limitation in 4)
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.